# Run v30: kanıtlı optimuma ulaşma süresi

**Protokol:** `experiments/PROTOCOL_v30.md`. Koşudan önce yazıldı; defter, protokol ve defter GitHub'da değilse başlamaz.

**Soru.** Optimal kenarları daha az kaçıran listeler, LKH-3'ün kanıtlı optimuma daha çabuk ulaşmasını sağlar mı?

**Örnekler (37).** Pilotun 18 sentetik örneği (optimumları Concorde ile kanıtlandı) ve yayımlanmış optimumu olan 19 TSPLIB örneği (1.002–2.392 düğüm).

**Kollar (7):**

- Kesintisiz: `pop` (POPMUSIC-5) ve `alpha` (LKH-3'ün varsayılanı).
- İlk denemeden sonra yeniden başlatılan: `pop_r`, `beta`, `count`, `mix`, `union`.

Kol başına 6 tohum kullanılır. Üst sınır düğüm başına 0,05 s LKH-3 işlemci saatidir.

**Ölçüt.** PAR10: optimuma ulaşma süresi; sınırda ulaşamayan koşu 10 × sınır sayılır. İki kolun PAR10 farkı sınırın %1'ini aşmıyorsa eşit sayılır.

**Hipotezler** (kesin işaret testi, Holm):

- **H1:** `beta` < `pop`.
- **H2:** `beta` < `alpha`.

**Çalıştırma.**

1. Protokolü ve defteri commit edip GitHub'a push edin.
2. Colab'da L4 GPU + Yüksek RAM (12 vCPU) seçin; T4 + 8 vCPU da olur.
3. *Tümünü çalıştır*. İlk hücre Drive'ı bağlar. Sonuçlar `MyDrive/BetaTour/v30_tto37` klasörüne yazılır.
4. Oturum düşerse aynı makine türüyle yeniden çalıştırın; biten koşular atlanır.

**Tahmini süre:**

| Adım | 12 vCPU | 8 vCPU |
|---|---|---|
| Kurulum | 3–5 dk | 3–5 dk |
| Öz sınama | 1–2 dk | 1–2 dk |
| İlk denemeler ve listeler | 4–8 dk | 5–10 dk |
| Koşular (1.554) | 3–5 saat | 4–7 saat |
| Analiz ve arşiv | 1–2 dk | 1–2 dk |

In [ ]:
from google.colab import drive; drive.mount("/content/drive")  # noqa: E702
# Run v30: kanıtlı optimuma ulaşma süresi (protokol: experiments/PROTOCOL_v30.md).
# Ortam: L4 GPU + Yüksek RAM (12 vCPU) önerilir; T4 + 8 vCPU da kabul edilir (süre uzar). GPU yalnızca BetaTour
# çıkarımı içindir. Bütün oturumlar aynı makine türünde olmalıdır (süre LKH-3'ün işlemci saatiyle ölçülür).
# Tahmini süre: kurulum 3-5 dk, öz sınama 1-2 dk, ilk denemeler ve listeler 4-10 dk, ana adım 12 vCPU'da 3-5 saat
# (8 vCPU'da 4-7 saat), analiz ve arşiv 1-2 dk. Oturum düşerse defteri baştan çalıştırın; biten koşular atlanır.
import os

EXPERIMENT_NAME = "v30_tto37"
SAVE_TO_DRIVE = True
REQUIRE_PUSHED_PROTOCOL = True     # kurulum, protokol ve defter GitHub'da (main) bu kurulumla aynı değilse durur
ALLOW_HARDWARE_CHANGE = False      # True: sonraki oturum başka makine türünde sürebilir (rapora not düşülür)

SEEDS = list(range(201, 207))      # altı tohum (çalışmadaki gibi); ilk denemeler ve bütün kollar
K_LIST = 5
K_UNION = 10
K_LEARNED = 3                      # mix: BetaTour'un ilk üçü (run v21-v23'teki listeler)
K_GRAPH = 16                       # BetaTour grafı: 16 en yakın komşu + topluluk kenarları
CAP_PER_NODE = 0.05                # üst sınır: düğüm başına LKH-3 işlemci saniyesi (TOTAL_TIME_LIMIT)
PAR_FACTOR = 10.0                  # PAR10: sınırda optimuma ulaşmayan koşu 10 x sınır sayılır
TIE_FRACTION = 0.01                # iki kolun PAR10 farkı sınırın %1'ini aşmıyorsa eşit sayılır (protokol)
JOB_ORDER_SEED = 3030              # bütün koşuların tek rastgele sırası (yük kollara eşit dağılır)
BOOTSTRAP_SEED = 30
ARMS = ["pop", "alpha", "pop_r", "beta", "count", "mix", "union"]

# Pilotun 18 sentetik örneği (certified_pilot): (grup, düğüm, üretilen örnek sayısı, aile, üretici tohumu, kullanılan
# sıra numaraları). Optimumlar pilotta Concorde ile kanıtlandı; dosyaların SHA-256 değerleri aynı kodla hesaplandı.
SYNTHETIC = [("uni1k", 1000, 5, "uniform", 15101, [0, 1, 2, 3, 4]),
             ("clu1k", 1000, 5, "clustered", 15202, [0, 1, 2, 3, 4]),
             ("uni2k", 2000, 5, "uniform", 15303, [1, 3, 4]),
             ("clu2k", 2000, 5, "clustered", 15404, [0, 1, 2, 3, 4])]
OPTIMA = {"cp1_uni1k_00": 22893008, "cp1_uni1k_01": 23299083, "cp1_uni1k_02": 23243281, "cp1_uni1k_03": 22488252,
          "cp1_uni1k_04": 23364800, "cp1_clu1k_00": 10139325, "cp1_clu1k_01": 9698285, "cp1_clu1k_02": 12844846,
          "cp1_clu1k_03": 15289751, "cp1_clu1k_04": 16172229, "cp1_uni2k_01": 32773730, "cp1_uni2k_03": 32335931,
          "cp1_uni2k_04": 32618292, "cp1_clu2k_00": 14235210, "cp1_clu2k_01": 16900417, "cp1_clu2k_02": 12243261,
          "cp1_clu2k_03": 20185266, "cp1_clu2k_04": 11765714}
PROBLEM_SHA256 = {
    "cp1_uni1k_00": "87568f46d4a0965410c71c64ccc25f3fabd1524a677e8a1c86707d102e6da676",
    "cp1_uni1k_01": "36cc1821f033b5fe1030eeb2a2639cb8b5b242d9218ac0902c1a96fc866aa507",
    "cp1_uni1k_02": "811fcba04fb23c7fd9d31f00ef88670008701543c028b55e6336008d7031c155",
    "cp1_uni1k_03": "2448de1fdaa8a6243969d9ae67c3da09959b1c9ea275f94da8de1184918b3844",
    "cp1_uni1k_04": "cca760a0929c4c1ab2783a86976b5ff6ec4d690a623bfb0b8d90e396136e366a",
    "cp1_clu1k_00": "916c16a3b8e2377cd422c117dcccdb551276e243a143fb425d831a0887c299e7",
    "cp1_clu1k_01": "dfc4393271f2b226bd22426bda15d2b31a58895cad5c7936aadd4c52f325535d",
    "cp1_clu1k_02": "b8270b97b147ebc58b9e781a8a7fc9598bf7737d26ce4ca4cd26dd8d3d50b264",
    "cp1_clu1k_03": "b40f02b2bdf15ed6381fb51cac2d7ac9812ea057b8d8aecef9d611104c93b02e",
    "cp1_clu1k_04": "0db90d52e2339909d5752027860adda7250e4e42c4d6cf67b520477d256b34b4",
    "cp1_uni2k_01": "883d2cb398a5dd03ced216cb0db1d652c552948ec4fc20d88fb24a92d138736c",
    "cp1_uni2k_03": "4c186483f84d7a2e382fad65fa51ddd0d1a24a1998ca21d0cb1027faaf7d5201",
    "cp1_uni2k_04": "53a3c28a260cc9f644e14d2ab7798d4734500f45102868e3bfda9ea6bf5cc034",
    "cp1_clu2k_00": "136d3c107f07c81c3ffb95793f448f3076ab5e08c93ad846eaa00e484c3e0a9a",
    "cp1_clu2k_01": "66bb41052e43b2de6077ba790860403c3c00047258ba075aea8dc864bc44d1da",
    "cp1_clu2k_02": "76bd375638cf2e25c368f01e9afd2b2df95448c10caa374b3833c8870fb91dec",
    "cp1_clu2k_03": "dba165ed5003f8db5478b3e2789605e3e13c6fa8ce6b3167cbe89234efc773d2",
    "cp1_clu2k_04": "a55e5d2875a1ae459c5d3dd7e2e5779a69d8f7f6e91ffe8167e8f8f842804a90"}
# Yayımlanmış optimumu olan TSPLIB örnekleri (EUC_2D, 1.002-2.392 düğüm); optimumlar tsplab.TSPLIB_OPT'tan.
TSPLIB_NAMES = ["pr1002", "u1060", "vm1084", "pcb1173", "d1291", "rl1304", "rl1323", "nrw1379", "fl1400", "u1432",
                "fl1577", "d1655", "vm1748", "u1817", "rl1889", "d2103", "u2152", "u2319", "pr2392"]
PILOT_DIR = "/content/drive/MyDrive/BetaTour/certpilot1"   # isteğe bağlı: pilotun optimal turları (kapsama analizi)
TSPLIB_TEST_OPTIMA = None          # yalnızca test kipinde: yerelde yazılan TSPLIB biçimli dosyaların optimumları
SELF_N = 200

REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
PROTOCOL_PATH = "experiments/PROTOCOL_v30.md"
NOTEBOOK_FILE = "notebooks/33_BetaTour_v30_time_to_optimum.ipynb"
PROTOCOL_SHA256 = "51f9c7f96639392b3f0d1f13437ba252e974fa56f9fd4b54d0ac98c540bd010d"        # protokol dosyasının özeti (defter kurulurken yazılır)
ALPHAENC_SHA256 = "58903205f379cf89e7f102333bac9bfc9abffd6ff816d5e4df3582dce75de495"        # alphaenc_v21.py (run v21'in modülü, değiştirilmeden)
TTO_SHA256 = "b1ece107596fc750b4ecc7bef4f3f1fe330dd4eafba4c0408913bfb5e1342d1d"                  # tto_v30.py (bu koşunun modülü)
NOTEBOOK_BUILD = "b251f08ae4f9124f96cac810fd1081e39da0ee95f5de839f808777246ed94519"          # bütün kod hücrelerinin, modüllerin ve protokolün özeti

TEST_MODE = os.environ.get("BETATOUR_V30_TEST") == "1"     # yalnızca yerel öz sınama; Colab'da kapalıdır
if TEST_MODE:
    EXPERIMENT_NAME = "v30_selftest"
    SAVE_TO_DRIVE = False
    REQUIRE_PUSHED_PROTOCOL = False
    import json as _json
    _test = _json.loads(open(os.environ["BETATOUR_V30_TEST_SPEC"]).read())
    SYNTHETIC = [tuple(x) for x in _test["synthetic"]]
    OPTIMA, PROBLEM_SHA256 = _test["optima"], _test["sha256"]
    TSPLIB_NAMES = _test["tsplib_names"]
    TSPLIB_TEST_OPTIMA = _test["tsplib_optima"]
    CAP_PER_NODE = float(_test.get("cap_per_node", CAP_PER_NODE))
    PILOT_DIR = _test.get("pilot_dir", "")
    SELF_N = 60
assert len(SEEDS) == 6 and K_LEARNED < K_LIST < K_UNION <= K_GRAPH

In [ ]:
%%writefile alphaenc_v21.py
"""Run v21: alpha-encoding control. Helper functions embedded in notebook 23.

A candidate file of LKH-3 holds, per node, the father of the node in the minimum 1-tree (Dad) and the candidates
with their alpha-values. LKH-3 uses both fields after it has read the file:

  * Every trial after the first starts from a tour built by a random walk (ChooseInitialTour.c). The walk keeps an
    edge of the best or next best tour only if that edge is a candidate whose alpha-value is zero (case C). With
    CANDIDATE_SET_TYPE = POPMUSIC and SUBGRADIENT = NO, LKH gives alpha = 0 to the edges of its minimum 1-tree, about
    two candidates per node.
  * Until the first tour of a run is recorded, the macro Near(a, b) is true for the edges of the minimum 1-tree
    (a is the father of b or b the father of a); the first trial does not break these edges first.

The candidate files of runs v1-v8 gave alpha = 0 to the first candidate of each node only (alpha = 0, 100, 200, ...)
and wrote no father, so their restarted arms kept fewer edges of the best tour at every trial.

This module holds
  * the two encodings compared in run v21 ("rank": 0, 100, 200, ..., no father; "tree": alpha = 0 exactly on the
    candidates that LKH's own lists of the same seed mark with alpha = 0, and the fathers of those lists),
  * LKH task functions that mirror tsplab.task_first_trial, task_anytime and task_guided, return the alpha-values
    and fathers of LKH's own lists and count the completed trials (TRACE_LEVEL = 2),
  * the share of a tour's edges that LKH's restart keeps under a given candidate file,
  * a self-check that compares LKH's search under its own candidate file and under rewritten files, and
  * the paired statistics of the analysis.
"""

from __future__ import annotations

import os
import re
import shutil
import tempfile
from math import comb

import numpy as np

import tsplab as T

VERSION = "v21-alphaenc-3"

# One line per completed trial at TRACE_LEVEL = 2: "* 12: Cost = 123, Time = 1.23 sec." (new best tour of the run)
# or "  13: Cost = 124, Time = 1.31 sec." (no improvement). LKH reports its own CPU time since the search began.
TRIAL_RX = re.compile(r"^(\*?)[ \t]*(\d+):[ \t]*Cost[ \t]*=[ \t]*(-?\d+)[^\n]*?Time[ \t]*=[ \t]*([\d.]+)[ \t]*sec", re.M)


# ----------------------------------------------------------------------------- encodings
def edge_keys(a, b, n):
    """Undirected edge keys min * n + max for 0-based node arrays a and b."""
    a = np.asarray(a, dtype=np.int64)
    b = np.asarray(b, dtype=np.int64)
    return np.minimum(a, b) * int(n) + np.maximum(a, b)


def tree_edge_keys(cand_native, alpha_native):
    """Keys of the edges that LKH's own lists mark with alpha = 0 (the edges of its minimum 1-tree).

    cand_native: (n, K) 0-based neighbour ids, -1 padded; alpha_native: (n, K) alpha-values written by LKH."""
    cand_native = np.asarray(cand_native, dtype=np.int64)
    alpha_native = np.asarray(alpha_native, dtype=np.int64)
    assert cand_native.shape == alpha_native.shape and cand_native.ndim == 2
    n = cand_native.shape[0]
    rows = np.broadcast_to(np.arange(n, dtype=np.int64)[:, None], cand_native.shape)
    mask = (cand_native >= 0) & (alpha_native == 0)
    return np.unique(edge_keys(rows[mask], cand_native[mask], n))


def rank_alpha(cand):
    """The encoding of runs v1-v8: alpha = 0, 100, 200, ... in list order (tsplab.write_candidates default)."""
    cand = np.asarray(cand)
    return np.broadcast_to(100 * np.arange(cand.shape[1], dtype=np.int64)[None, :], cand.shape).copy()


def tree_alpha(cand, tree_keys):
    """Tree encoding: alpha = 0 for the entries of `cand` whose edge is in `tree_keys`, and 100, 200, ... by list
    position for the other entries. LKH sorts each list by alpha and then by length when it reads the file, so the
    alpha = 0 entries move to the front and the other entries keep their order. Returns (alpha, is_tree)."""
    cand = np.asarray(cand, dtype=np.int64)
    n, k = cand.shape
    rows = np.broadcast_to(np.arange(n, dtype=np.int64)[:, None], cand.shape)
    valid = cand >= 0
    keys = edge_keys(rows, np.where(valid, cand, 0), n)
    is_tree = valid & np.isin(keys, np.asarray(tree_keys, dtype=np.int64))
    alpha = np.where(is_tree, 0, 100 * (np.arange(k, dtype=np.int64)[None, :] + 1))
    return alpha.astype(np.int64), is_tree


def kept_share(cand, alpha, tour):
    """Share of the edges of `tour` that LKH's restart can keep under a candidate file (cand, alpha): the edge (i, j)
    is kept when the walk reaches i and j is a candidate of i with alpha = 0. Mean over both walking directions."""
    cand = np.asarray(cand, dtype=np.int64)
    alpha = np.asarray(alpha, dtype=np.int64)
    tour = np.asarray(tour, dtype=np.int64)
    n = len(tour)
    assert cand.shape[0] == n and cand.shape == alpha.shape
    succ = np.empty(n, dtype=np.int64)
    pred = np.empty(n, dtype=np.int64)
    succ[tour] = np.roll(tour, -1)
    pred[tour] = np.roll(tour, 1)
    zero = (cand >= 0) & (alpha == 0)
    forward = ((cand == succ[:, None]) & zero).any(1).mean()
    backward = ((cand == pred[:, None]) & zero).any(1).mean()
    return float((forward + backward) / 2.0)


# ----------------------------------------------------------------------------- LKH tasks
def trial_times(stdout):
    """End times (LKH's clock, seconds since its search began) of the trials printed at TRACE_LEVEL = 2."""
    rows = [(int(k), float(t)) for _, k, _, t in TRIAL_RX.findall(stdout or "")]
    times = []
    for index, (k, t) in enumerate(rows, 1):
        if k != index:          # a truncated or foreign line: keep the consistent prefix only
            break
        times.append(t)
    return times


def trial_costs(stdout):
    """(trial number, cost) of every trial printed at TRACE_LEVEL = 2, in order."""
    return [(int(k), int(c)) for _, k, c, _ in TRIAL_RX.findall(stdout or "")]


def trials_within(times, offset, budget, fractions=(0.25, 0.5, 0.75, 1.0)):
    """Number of completed trials whose absolute end time (offset + LKH time) is within each fraction of the budget."""
    absolute = np.asarray(times, dtype=np.float64) + float(offset)
    return [int((absolute <= f * budget + 1e-9).sum()) for f in fractions]


def task_first_trial(lkh_bin, workroot, seed, xy, problem_file=None, ew="EUC_2D", k=T.CAND_K, timeout=None):
    """tsplab.task_first_trial (ascent-free POPMUSIC lists and one trial) at TRACE_LEVEL = 2 that also returns the
    alpha-values and the fathers (0-based, -1 for the root) of the run's own candidate lists, as LKH wrote them to
    its CANDIDATE_FILE. tsplab reads each list in ascending alpha order, which is the order of LKH's file."""
    p = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_TRIALS": 1, "MAX_CANDIDATES": int(k),
         "SEED": int(seed), "TRACE_LEVEL": 2}
    res = T.lkh_solve(lkh_bin, xy=None if problem_file else xy, problem_file=problem_file, ew=ew, params=p,
                      workroot=workroot, want_cands=True, timeout=timeout)
    n = len(xy)
    if not T.is_valid_tour(res["tour"], n):
        raise RuntimeError(f"first trial gave no valid tour (rc={res['returncode']}): {res['stdout'][-500:]}")
    if "cand" not in res:
        raise RuntimeError("LKH did not write the candidate file: " + res["stdout"][-500:])
    cand, alpha, dad = res["cand"]["cand"], res["cand"]["alpha"], np.asarray(res["cand"]["dad"], dtype=np.int64)
    if cand.shape != (n, k) or alpha.shape != (n, k) or dad.shape != (n,):
        raise RuntimeError(f"unexpected candidate list shape {cand.shape}")       # no silent truncation to k
    return {"tour": res["tour"].astype(np.int32), "cost": float(T.tour_length(xy, res["tour"], ew)),
            "cand": cand.astype(np.int32), "alpha": alpha.astype(np.int64), "dad": dad, "events": res["events"],
            "pre_time": res["pre_time"], "wall": res["wall"], "t_start": res["t_start"]}


def task_anytime(lkh_bin, workroot, budget, seed, xy=None, problem_file=None, ew="EUC_2D", params=None,
                 timeout=None):
    """tsplab.task_anytime with TRACE_LEVEL = 2; also returns the end time of every completed trial. At this trace
    level POPMUSIC prints its own "Cost.min" and "Cand.min" lines first, so tsplab's cost_min and cand_stats would
    describe the POPMUSIC sample tours; they are not returned."""
    p = {"TIME_LIMIT": float(budget), "MAX_TRIALS": 100_000_000, "SEED": int(seed), "TRACE_LEVEL": 2}
    p.update(params or {})
    res = T.lkh_solve(lkh_bin, xy=xy, problem_file=problem_file, ew=ew, params=p, workroot=workroot,
                      timeout=budget * 3 + 600 if timeout is None else timeout, keep_stdout=True)
    out = {k: res.get(k) for k in ("events", "pre_time", "wall", "returncode", "timed_out", "t_start")}
    out["trial_times"] = trial_times(res.get("stdout"))
    return out


def task_guided(lkh_bin, workroot, budget, seed, xy, cand, alpha=None, dad=None, init_tour=None, problem_file=None,
                ew="EUC_2D", extra_params=None, timeout=None):
    """tsplab.task_guided without fixed edges (candidate file with the given alpha-values and fathers, zero
    penalties, initial tour, no ascent) with TRACE_LEVEL = 2; also returns the end time of every completed trial.
    alpha = None writes 0, 100, 200, ... and dad = None writes no father, as in runs v1-v8."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="g_")
    try:
        n = len(xy)
        pf = problem_file
        if pf is None:
            pf = os.path.join(wd, "p.tsp")
            T.write_tsp(pf, xy, ew=ew)
        cf, pif = os.path.join(wd, "g.cand"), os.path.join(wd, "g.pi")
        T.write_candidates(cf, cand, alpha, dad)
        T.write_pi(pif, np.zeros(n))
        params = {"SUBGRADIENT": "NO", "TIME_LIMIT": float(budget), "MAX_TRIALS": 100_000_000,
                  "SEED": int(seed), "MAX_CANDIDATES": int(np.asarray(cand).shape[1]), "TRACE_LEVEL": 2}
        if init_tour is not None:
            tp = os.path.join(wd, "init.tour")
            T.write_tour(tp, init_tour)
            params["INITIAL_TOUR_FILE"] = tp
        params.update(extra_params or {})
        params["CANDIDATE_FILE"], params["PI_FILE"] = cf, pif
        res = T.run_lkh(lkh_bin, pf, params, os.path.join(wd, "run"),
                        timeout=budget * 3 + 600 if timeout is None else timeout, keep_stdout=True)
        ok = T.is_valid_tour(res["tour"], n)
        out = {k: res.get(k) for k in ("events", "pre_time", "wall", "returncode", "timed_out", "t_start")}
        out["cost_check"] = float(T.tour_length(xy, res["tour"], ew)) if ok else None
        out["trial_times"] = trial_times(res.get("stdout"))
        if not ok:
            out["stdout_tail"] = (res.get("stdout") or "")[-800:]
        return out
    finally:
        shutil.rmtree(wd, ignore_errors=True)


def encoding_selfcheck(lkh_bin, workroot, xy, seed, trials=25, k=T.CAND_K, ew="EUC_2D", timeout=300):
    """Compares, on the machine of the run, LKH's search under the candidate file it wrote itself with its search
    under four rewritten files: LKH's own alpha-values or 0, 100, 200, ..., each with and without the fathers.
    All continuations start from the tour of the first trial, have a fixed number of trials and no time limit, so
    their traces are deterministic. Expected from the source code:
      * own alpha-values with fathers: the same cost at every trial as under LKH's own file;
      * 0, 100, 200, ... with fathers: a different search from the second trial on (restart walk, case C);
      * without fathers: a difference, if any, can start in the first trial (Near() before a tour is recorded).
    Also checks that the tree rule reproduces the entries that LKH marks with alpha = 0."""
    os.makedirs(workroot or tempfile.gettempdir(), exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="enc_")
    try:
        n = len(xy)
        pf, own = os.path.join(wd, "p.tsp"), os.path.join(wd, "own.cand")
        T.write_tsp(pf, xy, ew=ew)
        first = T.run_lkh(lkh_bin, pf, {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_TRIALS": 1,
                                        "MAX_CANDIDATES": int(k), "SEED": int(seed), "TRACE_LEVEL": 2,
                                        "CANDIDATE_FILE": own}, os.path.join(wd, "first"), timeout=timeout)
        if first["returncode"] != 0 or not T.is_valid_tour(first["tour"], n) or not os.path.exists(own):
            raise RuntimeError("self-check: the first trial failed: " + (first.get("stdout") or "")[-500:])
        lists = T.read_candidates(own, k_pad=k)
        cand, alpha, dad = lists["cand"], lists["alpha"], lists["dad"]
        if cand.shape != (n, k):
            raise RuntimeError(f"self-check: unexpected candidate list shape {cand.shape}")
        pif, tp = os.path.join(wd, "zero.pi"), os.path.join(wd, "first.tour")
        T.write_pi(pif, np.zeros(n))
        T.write_tour(tp, first["tour"])
        variants = {"native_dad": (alpha, dad), "native_nodad": (alpha, None),
                    "rank_dad": (rank_alpha(cand), dad), "rank_nodad": (rank_alpha(cand), None)}
        files = {"lkh_file": own}
        for name, (a, d) in variants.items():
            files[name] = os.path.join(wd, name + ".cand")
            T.write_candidates(files[name], cand, a, d)
        traces = {}
        for name, cf in files.items():
            res = T.run_lkh(lkh_bin, pf, {"SUBGRADIENT": "NO", "MAX_TRIALS": int(trials), "MAX_CANDIDATES": int(k),
                                          "SEED": int(seed), "TRACE_LEVEL": 2, "INITIAL_TOUR_FILE": tp,
                                          "CANDIDATE_FILE": cf, "PI_FILE": pif}, os.path.join(wd, name),
                            timeout=timeout, keep_stdout=True)
            traces[name] = trial_costs(res.get("stdout"))
            if res["returncode"] != 0 or [k_ for k_, _ in traces[name]] != list(range(1, int(trials) + 1)):
                raise RuntimeError(f"self-check: continuation '{name}' failed: " + (res.get("stdout") or "")[-500:])

        def first_difference(name):
            trials_ = [k_ for (k_, a), (_, b) in zip(traces[name], traces["lkh_file"]) if a != b]
            return int(trials_[0]) if trials_ else None

        valid = cand >= 0
        _, is_tree = tree_alpha(cand, tree_edge_keys(cand, alpha))
        return {"nodes": int(n), "trials": int(trials), "seed": int(seed),
                "native_file_equals_lkh_file": traces["native_dad"] == traces["lkh_file"],
                "first_trial_with_a_different_cost": {name: first_difference(name) for name in variants},
                "tree_rule_matches_lkh": bool(np.array_equal(is_tree, valid & (alpha == 0))),
                "candidates_per_node": float(valid.sum() / n),
                "zero_alpha_per_node": float((valid & (alpha == 0)).sum() / n),
                "nodes_with_a_father": int((dad >= 0).sum()),
                "kept_share_native": kept_share(cand, alpha, first["tour"]),
                "kept_share_rank": kept_share(cand, rank_alpha(cand), first["tour"]),
                "best_cost": {name: int(min(c for _, c in trace)) for name, trace in traces.items()}}
    finally:
        shutil.rmtree(wd, ignore_errors=True)


# ----------------------------------------------------------------------------- statistics
def sign_test_p(wins, losses):
    """Exact two-sided sign test."""
    n, k = int(wins) + int(losses), min(int(wins), int(losses))
    if n == 0:
        return 1.0
    return float(min(1.0, 2.0 * sum(comb(n, j) for j in range(k + 1)) / 2.0 ** n))


def paired_stats(values, bootstrap_seed, replicates=5000):
    """Paired percentage differences (first minus second arm): wins are negative values (the first arm is shorter)."""
    x = np.asarray(values, dtype=np.float64)
    assert x.ndim == 1 and len(x) and np.isfinite(x).all()
    wins, losses = int((x < -1e-12).sum()), int((x > 1e-12).sum())
    rng = np.random.default_rng(bootstrap_seed)
    means = x[rng.integers(0, len(x), size=(replicates, len(x)))].mean(axis=1)
    low, high = np.quantile(means, [0.025, 0.975])
    return dict(instances=int(len(x)), win=wins, tie=int(len(x) - wins - losses), loss=losses,
                mean_delta_pct=float(x.mean()), median_delta_pct=float(np.median(x)),
                ci95_low_pct=float(low), ci95_high_pct=float(high), sign_p=sign_test_p(wins, losses))


def holm(p_values):
    """Holm-adjusted p-values in the input order."""
    p = np.asarray(p_values, dtype=np.float64)
    order = np.argsort(p, kind="stable")
    adjusted = np.empty(len(p))
    running = 0.0
    for rank, j in enumerate(order):
        running = max(running, (len(p) - rank) * p[j])
        adjusted[j] = min(1.0, running)
    return [float(v) for v in adjusted]


In [ ]:
%%writefile tto_v30.py
"""Run v30: time to the proven optimum with different candidate lists. Helper functions embedded in notebook 33.

LKH-3.0.14 stops a run when its tour reaches OPTIMUM (STOP_AT_OPTIMUM = YES) or when its CPU clock, counted from the
start of the program, reaches TOTAL_TIME_LIMIT. The time to the optimum of a run is LKH-3's Time.total; for the
restarted arms the pause before the restart is added. This module holds

  * the first trial of a seed (ascent-free POPMUSIC-5, one trial), with its lists, alpha-values, fathers and CPU time;
  * the candidate files of the restarted arms (pop_r, beta, count, mix, union) and the time to build them;
  * one LKH-3 run that stops at the optimum or at the cap, with or without a candidate file and an initial tour;
  * the time to the optimum, PAR scores, sign tests with Holm correction and bootstrap intervals.

The candidate-file encodings come from run v21 (alphaenc.py, unchanged).
"""

from __future__ import annotations

import os
import shutil
import tempfile
import time

import numpy as np

import tsplab as T

try:
    import alphaenc_v21 as A        # the notebook writes run v21's module under this name
except ImportError:                 # pragma: no cover - repository layout (tests)
    import alphaenc as A

VERSION = "v30-tto-1"
ARMS = ("pop", "alpha", "pop_r", "beta", "count", "mix", "union")
UNINTERRUPTED = {"pop": {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO"},
                 "alpha": {"CANDIDATE_SET_TYPE": "ALPHA", "SUBGRADIENT": "YES"}}
RESTARTED = ("pop_r", "beta", "count", "mix", "union")
MODEL_ARMS = ("beta", "mix", "union")
MAX_TRIALS = 100_000_000


# ----------------------------------------------------------------------------------------------------- LKH runs
def first_trial(lkh_bin, workroot, seed, xy, problem_file, ew, optimum, k=5, timeout=900):
    """One ascent-free POPMUSIC-k trial (the first trial of `pop` with the same seed). Returns the tour, its length,
    the run's own lists with LKH-3's alpha-values and fathers, and LKH-3's CPU time (Time.total)."""
    params = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_TRIALS": 1, "MAX_CANDIDATES": int(k),
              "SEED": int(seed), "TRACE_LEVEL": 1, "OPTIMUM": int(optimum), "STOP_AT_OPTIMUM": "YES"}
    res = T.lkh_solve(lkh_bin, problem_file=problem_file, ew=ew, params=params, workroot=workroot, want_cands=True,
                      timeout=timeout)
    n = len(xy)
    if res["returncode"] != 0 or not T.is_valid_tour(res["tour"], n):
        raise RuntimeError(f"first trial failed (seed {seed}, rc={res['returncode']}): {res['stdout'][-600:]}")
    if "cand" not in res or res["time_total"] is None:
        raise RuntimeError("first trial: no candidate file or no Time.total: " + res["stdout"][-600:])
    cand, alpha, dad = res["cand"]["cand"], res["cand"]["alpha"], np.asarray(res["cand"]["dad"], dtype=np.int64)
    if cand.shape != (n, k) or alpha.shape != (n, k) or dad.shape != (n,):
        raise RuntimeError(f"first trial: unexpected list shape {cand.shape}")
    return {"tour": np.asarray(res["tour"], dtype=np.int32), "cost": float(T.tour_length(xy, res["tour"], ew)),
            "cand": cand.astype(np.int32), "alpha": alpha.astype(np.int64), "dad": dad,
            "time_total": float(res["time_total"]), "wall": float(res["wall"]), "pre_time": res["pre_time"]}


def lkh_run(lkh_bin, workroot, problem_file, xy, ew, seed, optimum, cpu_limit, params=None, files=None,
            init_tour=None, timeout=None, max_trials=MAX_TRIALS, return_tour=False):
    """One LKH-3 run that stops at the optimum or when LKH-3's CPU clock reaches cpu_limit (TOTAL_TIME_LIMIT).
    files = (cand, alpha, dad) gives a candidate file with zero penalties and no ascent (as in run v21); init_tour
    starts the first trial from that tour. Returns the recomputed length of the output tour, whether it is the
    optimum, LKH-3's Time.total (CPU seconds), the wall time and the number of trials (and the tour if asked)."""
    n = len(xy)
    os.makedirs(workroot, exist_ok=True)
    wd = tempfile.mkdtemp(dir=workroot, prefix="r_")
    try:
        p = dict(params or {})
        if files is not None:
            cand, alpha, dad = files
            cand = np.asarray(cand)
            cf, pif = os.path.join(wd, "g.cand"), os.path.join(wd, "g.pi")
            T.write_candidates(cf, cand, alpha, dad)
            T.write_pi(pif, np.zeros(n))
            p.update(CANDIDATE_FILE=cf, PI_FILE=pif, SUBGRADIENT="NO", MAX_CANDIDATES=int(cand.shape[1]))
        else:
            p.setdefault("MAX_CANDIDATES", 5)
        if init_tour is not None:
            tp = os.path.join(wd, "init.tour")
            T.write_tour(tp, init_tour)
            p["INITIAL_TOUR_FILE"] = tp
        limit = max(float(cpu_limit), 0.05)
        p.update(RUNS=1, MAX_TRIALS=int(max_trials), SEED=int(seed), OPTIMUM=int(optimum), STOP_AT_OPTIMUM="YES",
                 TOTAL_TIME_LIMIT=round(limit, 3), TRACE_LEVEL=1)
        res = T.run_lkh(lkh_bin, problem_file, p, os.path.join(wd, "run"),
                        timeout=timeout if timeout is not None else 3.0 * limit + 120.0, keep_stdout=True)
        ok = T.is_valid_tour(res["tour"], n)
        cost = float(T.tour_length(xy, res["tour"], ew)) if ok else None
        stdout = res.get("stdout") or ""
        out = {"cost": cost, "found": bool(ok and cost == float(optimum)), "shorter": bool(ok and cost < optimum),
               "time_total": None if res["time_total"] is None else float(res["time_total"]),
               "wall": float(res["wall"]), "trials": res["trials"], "pre_time": res["pre_time"],
               "returncode": res["returncode"], "timed_out": bool(res["timed_out"]), "cost_min": res["cost_min"],
               "new_optimum": "New OPTIMUM" in stdout, "cpu_limit": limit}
        if not ok or res["returncode"] != 0 or res["time_total"] is None:
            out["stdout_tail"] = stdout[-1500:] + (res.get("stderr") or "")[-500:]
        if return_tour:
            out["tour"] = np.asarray(res["tour"], dtype=np.int64) if ok else None
        return out
    finally:
        shutil.rmtree(wd, ignore_errors=True)


# ------------------------------------------------------------------------------------------------ candidate files
def _valid(cand, n, k_max):
    cand = np.asarray(cand)
    assert cand.ndim == 2 and cand.shape[0] == n and cand.shape[1] <= k_max, cand.shape
    assert np.all((cand >= -1) & (cand < n)), "invalid candidate"
    assert not np.any(cand == np.arange(n)[:, None]), "a node is its own candidate"
    for i in range(cand.shape[1]):
        for j in range(i + 1, cand.shape[1]):
            assert not np.any((cand[:, i] == cand[:, j]) & (cand[:, i] >= 0)), "repeated candidate"
    assert np.all((cand >= 0).sum(1) >= 2), "a node with fewer than two candidates"


def arm_lists(firsts, prediction, k=5, k_union=10, k_learned=3):
    """Candidate files (cand, alpha, dad) of the restarted arms for every seed, the time to build each arm's lists
    (wall seconds; the inference of the model is not included) and the long rankings (BetaTour and counting).

    pop_r : the seed's own POPMUSIC-k lists with LKH-3's alpha-values and fathers (its own file).
    beta  : BetaTour's top k; count: top k by counting over the first-trial tours on BetaTour's graph (ties by
            length); mix: BetaTour's top k_learned, then the seed's POPMUSIC-k, then BetaTour's next ones (runs v21
            to v23); union: BetaTour's top k, then the seed's POPMUSIC-k entries not yet listed (up to k_union).
            All four in the tree encoding of run v21 (alpha = 0 on the edges of the seed's own minimum 1-tree) with
            the fathers of the seed's own file."""
    W = len(firsts)
    n = len(firsts[0]["tour"])
    rank = np.asarray(prediction["rank"], dtype=np.int64)
    NB = np.asarray(prediction["NB"], dtype=np.int64)
    D = np.asarray(prediction["D"], dtype=np.float64)
    assert rank.shape == NB.shape == D.shape and rank.shape[0] == n and rank.shape[1] >= k_union
    start = time.perf_counter()
    succ = np.stack([T.tour_succ_pred(f["tour"])[0] for f in firsts], axis=1)
    tree_keys = [A.tree_edge_keys(f["cand"], f["alpha"]) for f in firsts]
    common = time.perf_counter() - start

    lists, build = {arm: [] for arm in RESTARTED}, {}
    start = time.perf_counter()
    for f in firsts:
        lists["pop_r"].append((f["cand"].astype(np.int64), f["alpha"].astype(np.int64), f["dad"]))
    build["pop_r"] = time.perf_counter() - start

    start = time.perf_counter()
    top = rank[:, :k]
    for j, f in enumerate(firsts):
        lists["beta"].append((top, A.tree_alpha(top, tree_keys[j])[0], f["dad"]))
    build["beta"] = common + time.perf_counter() - start

    start = time.perf_counter()
    count_rank = T.frequency_candidates(NB, D, T.ensemble_frequency(NB, succ), k=NB.shape[1]).astype(np.int64)
    ctop = count_rank[:, :k]
    for j, f in enumerate(firsts):
        lists["count"].append((ctop, A.tree_alpha(ctop, tree_keys[j])[0], f["dad"]))
    build["count"] = common + time.perf_counter() - start

    start = time.perf_counter()
    for j, f in enumerate(firsts):
        mix = T.mix_candidates(rank, f["cand"], k_first=k_learned, k=k, fill=rank[:, k_learned:]).astype(np.int64)
        lists["mix"].append((mix, A.tree_alpha(mix, tree_keys[j])[0], f["dad"]))
    build["mix"] = common + time.perf_counter() - start

    start = time.perf_counter()
    for j, f in enumerate(firsts):
        uni = T.mix_candidates(rank, f["cand"], k_first=k, k=k_union).astype(np.int64)
        lists["union"].append((uni, A.tree_alpha(uni, tree_keys[j])[0], f["dad"]))
    build["union"] = common + time.perf_counter() - start

    for arm in RESTARTED:
        assert len(lists[arm]) == W
        for cand, alpha, dad in lists[arm]:
            _valid(cand, n, k_union if arm == "union" else k)
            assert alpha.shape == cand.shape and dad.shape == (n,)
    return lists, {arm: float(v) for arm, v in build.items()}, rank, count_rank


def pauses(first_cpu, model_s, build_s):
    """Pause of each restarted arm: the longest CPU time of the first trials (they run in parallel), plus the
    inference of the model for the arms that use it, plus the time to build the arm's lists."""
    longest = float(np.max(first_cpu))
    return {arm: longest + (float(model_s) if arm in MODEL_ARMS else 0.0) + float(build_s[arm]) for arm in RESTARTED}


def missed_optimal_edges(cand, opt_tour):
    """Edges of the optimal tour that are in neither end's list."""
    cand = np.asarray(cand)
    return int(T.candidate_coverage(cand, opt_tour, ks=[cand.shape[1]])[cand.shape[1]]["missing"])


# ------------------------------------------------------------------------------------------------------ scores
def time_to_optimum(record, cap):
    """Seconds to the optimum of one run record, or inf when the run did not reach it within the cap."""
    if record.get("first_found"):
        t = float(record["first_cpu"])
    elif record.get("found") and record.get("time_total") is not None:
        t = float(record.get("pause", 0.0)) + float(record["time_total"])
    else:
        return float("inf")
    return t if t <= cap + 1e-9 else float("inf")


def par(times, cap, factor=10.0):
    t = np.asarray(times, dtype=np.float64)
    return float(np.where(np.isfinite(t), t, factor * cap).mean())


def holm(pvalues):
    """Holm-adjusted p-values (same order as the input)."""
    p = np.asarray(pvalues, dtype=np.float64)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[i]))
        adjusted[i] = running
    return adjusted


def compare(first, second, tol=0.0, rel_tol=1e-9):
    """Paired comparison of two arrays (lower is better): wins of the first, ties, losses, exact two-sided sign p.
    Differences of at most tol (a number or one value per pair) count as ties."""
    a = np.asarray(first, dtype=np.float64)
    b = np.asarray(second, dtype=np.float64)
    tie = np.isclose(a, b, rtol=rel_tol, atol=1e-9) | (np.abs(a - b) <= np.asarray(tol, dtype=np.float64))
    wins, losses = int(((a < b) & ~tie).sum()), int(((a > b) & ~tie).sum())
    return {"wins": wins, "ties": int(tie.sum()), "losses": losses, "p": A.sign_test_p(wins, losses)}


def ratio_interval(first, second, seed, replicates=5000):
    """Geometric mean of first/second over instances and a 95% percentile interval (bootstrap over instances)."""
    r = np.log(np.asarray(first, dtype=np.float64) / np.asarray(second, dtype=np.float64))
    rng = np.random.default_rng(seed)
    boot = r[rng.integers(0, len(r), size=(replicates, len(r)))].mean(1)
    return {"ratio": float(np.exp(r.mean())), "low": float(np.exp(np.quantile(boot, 0.025))),
            "high": float(np.exp(np.quantile(boot, 0.975)))}


In [ ]:
# Kurulum (3-5 dk): depo (sabit commit), protokolün GitHub denetimi, iki modülün denetimi, LKH-3.0.14, BetaTour.
import hashlib
import json
import math
import platform
import shutil
import subprocess
import sys
import tarfile
import threading
import time
import traceback
import urllib.request
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import torch

CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
PARALLEL = max(1, CPU_COUNT // 2)          # protokol: görünen vCPU'ların yarısı kadar LKH süreci aynı anda
assert TEST_MODE or CPU_COUNT >= 8, f"En az 8 vCPU gerekli (Yüksek RAM seçin); görünen: {CPU_COUNT}"


def cpu_model_name():
    path = Path("/proc/cpuinfo")
    if path.exists():
        for line in path.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"


WORK_ROOT = Path(os.environ.get("BETATOUR_V30_WORK", "/content/betatour_v30_work"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Depo: run v27-v29 ile aynı sabit commit (örnek üreticisi, TSPLIB ve LKH yardımcıları, BetaTour ve model).
if TEST_MODE and os.environ.get("BETATOUR_V30_REPO"):
    REPO = Path(os.environ["BETATOUR_V30_REPO"])
else:
    REPO = WORK_ROOT / "repo"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git",
                        str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
    assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M

# Protokol koşudan önce GitHub'da olmalı: main dalındaki protokol bu defterin kurulduğu protokolle bayt bayt aynı
# değilse ya da GitHub'daki defter bu kurulum değilse kurulum durur.
PROTOCOL = {"path": PROTOCOL_PATH, "sha256": PROTOCOL_SHA256, "github_head": None, "protocol_commit": None,
            "protocol_commit_time": None, "notebook_on_github": None, "checked_utc": None}
if REQUIRE_PUSHED_PROTOCOL:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", "main"], check=True)
    github_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
    shown = subprocess.run(["git", "-C", str(REPO), "show", f"{github_head}:{PROTOCOL_PATH}"],
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    assert shown.returncode == 0, \
        f"Protokol GitHub'da yok ({PROTOCOL_PATH}). Önce depoyu commit ve push edin, sonra defteri yeniden çalıştırın."
    assert hashlib.sha256(shown.stdout).hexdigest() == PROTOCOL_SHA256, \
        "GitHub'daki protokol bu defterin protokolüyle aynı değil. Protokol değiştiyse defter yeniden kurulmalıdır."
    pushed = subprocess.run(["git", "-C", str(REPO), "show", f"{github_head}:{NOTEBOOK_FILE}"],
                            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    assert pushed.returncode == 0 and NOTEBOOK_BUILD.encode() in pushed.stdout, \
        "GitHub'daki defter bu defterle aynı kurulum değil. Depodaki defteri push edip onu çalıştırın."
    last = subprocess.check_output(["git", "-C", str(REPO), "log", "-1", "--format=%H %cI", github_head, "--",
                                    PROTOCOL_PATH], text=True).split()
    PROTOCOL.update(github_head=github_head, protocol_commit=last[0], protocol_commit_time=last[1],
                    notebook_on_github=True, checked_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    print("Protokol GitHub'da:", PROTOCOL_PATH, "| commit", last[0][:10], last[1])


def checked_module(file_name, expected):
    """Önceki hücrelerin yazdığı modül; içerik depodakiyle bayt bayt aynı olmalıdır (dosya sonundaki satır sonu sayısı
    arayüze göre değişebildiği için tek satır sonuna indirilir)."""
    path = Path(file_name).resolve()
    assert path.exists(), f"Önce {file_name} dosyasını yazan hücreyi çalıştırın."
    data = path.read_bytes().rstrip(b"\n") + b"\n"
    digest = hashlib.sha256(data).hexdigest()
    assert digest == expected, f"{file_name} beklenen modülle aynı değil; o hücreyi değiştirmeden çalıştırın."
    if path.read_bytes() != data:
        path.write_bytes(data)
    return path, digest


ALPHAENC_PATH, _ = checked_module("alphaenc_v21.py", ALPHAENC_SHA256)
TTO_PATH, _ = checked_module("tto_v30.py", TTO_SHA256)
sys.path.insert(0, str(TTO_PATH.parent))
import alphaenc_v21 as A
import tto_v30 as V

if SAVE_TO_DRIVE:
    OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / EXPERIMENT_NAME
for sub in ["instances", "runs", "summary", "code", "errors", "selfcheck"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
shutil.copy2(ALPHAENC_PATH, OUTPUT_ROOT / "code" / "alphaenc_v21.py")
shutil.copy2(TTO_PATH, OUTPUT_ROOT / "code" / "tto_v30.py")


def run_build(cmd, cwd, what):
    build = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=1800)
    if build.returncode != 0:
        raise RuntimeError(what + " derlemesi başarısız:\n" + build.stdout[-3000:])


def get_lkh_3014():
    """Değiştirilmemiş LKH-3.0.14: resmi kaynak; ulaşılamazsa SHA256 denetimli sabit kopya (önceki koşulardaki gibi)."""
    if TEST_MODE and os.environ.get("BETATOUR_V30_LKH"):
        return os.environ["BETATOUR_V30_LKH"], "local test binary"
    root = WORK_ROOT / "lkh"
    binary = root / "LKH-3.0.14" / "LKH"
    marker = root / "source.txt"
    if binary.exists() and marker.exists():
        return str(binary), marker.read_text().strip()
    root.mkdir(parents=True, exist_ok=True)
    options = [("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
               ("pinned copy", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar",
                "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33", 90)]
    last_error = None
    for label, url, expected_sha, timeout in options:
        archive = root / ("LKH-3.0.14.tar" if url.endswith(".tar") else "LKH-3.0.14.tgz")
        try:
            print("LKH kaynağı:", label, url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, archive.open("wb") as f:
                shutil.copyfileobj(response, f)
            if expected_sha:
                got = hashlib.sha256(archive.read_bytes()).hexdigest()
                assert got == expected_sha, f"LKH kopyası SHA256 uyuşmazlığı: {got}"
            shutil.rmtree(root / "LKH-3.0.14", ignore_errors=True)
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(root, filter="data")
                except TypeError:
                    tar.extractall(root)
            run_build(["make", "-j", str(min(4, CPU_COUNT))], root / "LKH-3.0.14", "LKH")
            assert binary.exists(), "LKH ikilisi oluşmadı"
            marker.write_text(f"{label}: {url}")
            return str(binary), f"{label}: {url}"
        except Exception as exc:
            last_error = exc
            print(f"{label} başarısız: {exc}")
    raise RuntimeError(f"LKH 3.0.14 indirilemedi veya derlenemedi: {last_error}")


t_setup = time.time()
LKH, LKH_SOURCE = get_lkh_3014()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert TEST_MODE or DEVICE.type == "cuda", "GPU gerekli (L4 ya da T4): BetaTour çıkarımı GPU'da yapılır."
checkpoint_path = REPO / "checkpoints" / "betatour_v2.pt"
assert hashlib.sha256(checkpoint_path.read_bytes()).hexdigest() == CHECKPOINT_SHA256, \
    "Model dosyası beklenen SHA256 değeriyle eşleşmiyor."
MODEL, CK = M.load_model(str(checkpoint_path), DEVICE)      # sabitlenmiş, güvenilen depo
assert MODEL.cfg.get("use_ensemble") is True

HARDWARE = {"device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none",
            "vcpus_visible": CPU_COUNT, "parallel_lkh": PARALLEL, "cpu_model": cpu_model_name(),
            "ram_gb": T.env_info().get("ram_gb"), "python": platform.python_version(), "numpy": np.__version__,
            "torch": torch.__version__}
MANIFEST = {"schema": 1, "experiment_name": EXPERIMENT_NAME, "test_mode": TEST_MODE, "protocol": PROTOCOL,
            "code_version": V.VERSION, "alphaenc_version": A.VERSION, "alphaenc_sha256": ALPHAENC_SHA256,
            "tto_sha256": TTO_SHA256, "notebook_build": NOTEBOOK_BUILD, "repo_commit": REPO_COMMIT,
            "checkpoint_sha256": CHECKPOINT_SHA256, "lkh_source": LKH_SOURCE,
            "design": {"seeds": SEEDS, "arms": ARMS, "k_list": K_LIST, "k_union": K_UNION, "k_learned": K_LEARNED,
                       "k_graph": K_GRAPH, "cap_per_node": CAP_PER_NODE, "par_factor": PAR_FACTOR,
                       "tie_fraction": TIE_FRACTION,
                       "job_order_seed": JOB_ORDER_SEED, "synthetic": [list(x) for x in SYNTHETIC],
                       "optima": OPTIMA, "problem_sha256": PROBLEM_SHA256, "tsplib": TSPLIB_NAMES},
            "hardware": HARDWARE}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in ["schema", "design", "alphaenc_sha256", "tto_sha256", "notebook_build", "repo_commit",
                "checkpoint_sha256"]:
        assert saved[key] == MANIFEST[key], f"{key} değişti. Ayrı bir EXPERIMENT_NAME kullanın."
    same = (saved["hardware"]["cpu_model"] == HARDWARE["cpu_model"]
            and saved["hardware"]["vcpus_visible"] == HARDWARE["vcpus_visible"])
    assert ALLOW_HARDWARE_CHANGE or same, \
        (f"Makine türü değişti ({saved['hardware']['cpu_model']}, {saved['hardware']['vcpus_visible']} vCPU -> "
         f"{HARDWARE['cpu_model']}, {HARDWARE['vcpus_visible']} vCPU). Aynı türü seçin; bulunamıyorsa ilk hücrede "
         "ALLOW_HARDWARE_CHANGE = True yapın (rapora not düşülür).")
    if saved["hardware"] != HARDWARE:
        print("UYARI: donanım veya kütüphane sürümleri ilk oturumdan farklı; her koşunun donanımı ayrıca kaydedilir.")
else:
    manifest_path.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))

print(f"Hazır ({time.time() - t_setup:.0f} s): {CPU_COUNT} vCPU, aynı anda {PARALLEL} LKH süreci | "
      f"{HARDWARE['cpu_model']} | {HARDWARE['gpu']} | LKH: {LKH_SOURCE} | modüller {V.VERSION}, {A.VERSION} | "
      f"sonuçlar: {OUTPUT_ROOT}")

In [ ]:
# Örnekler (1 dk): pilotun 18 sentetik örneği (yeniden üretilir, dosyaların SHA-256 değeri denetlenir; optimumlar
# pilotta Concorde ile kanıtlandı) ve yayımlanmış optimumu olan 19 TSPLIB örneği. Pilotun klasörü Drive'da varsa
# kaydettiği optimal turlar da okunur (yalnızca kapsama analizi için; uzunlukları denetlenir).
def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def build_instances():
    problems = WORK_ROOT / "problems"
    problems.mkdir(exist_ok=True)
    out = []
    for group, n, count, family, seed, keep in SYNTHETIC:
        rng = np.random.default_rng(seed)
        for i in range(count):
            xy = T.to_int_coords(T.gen_instance(n, family, rng))
            if i not in keep:
                continue
            name = f"cp1_{group}_{i:02d}"
            pf = problems / f"{name}.tsp"
            T.write_tsp(pf, xy, name=name, ew="EUC_2D")
            assert sha256_of(pf) == PROBLEM_SHA256[name], f"{name}: örnek dosyası pilottakiyle aynı değil"
            out.append(dict(name=name, group=group, family=family, n=n, xy=xy, ew="EUC_2D", problem_file=str(pf),
                            optimum=int(OPTIMA[name]), optimum_source="Concorde (certified pilot)"))
    if TEST_MODE:
        paths = dict(_test["tsplib_files"])
    else:
        paths = T.download_tsplib(str(WORK_ROOT / "tsplib"), names=TSPLIB_NAMES)
        absent = sorted(set(TSPLIB_NAMES) - set(paths))
        assert not absent, f"Eksik TSPLIB dosyaları: {absent}. Ağ bağlantısını denetleyip bu hücreyi yeniden çalıştırın."
    for name in TSPLIB_NAMES:
        data = T.read_tsplib(paths[name])
        n, ew = int(data["dimension"]), data["edge_weight_type"]
        assert len(data["xy"]) == n and ew == "EUC_2D", (name, n, ew)
        pf = problems / f"{name}.tsp"
        shutil.copy2(paths[name], pf)
        optimum = TSPLIB_TEST_OPTIMA[name] if TEST_MODE else T.TSPLIB_OPT[name]
        out.append(dict(name=name, group="tsplib", family="tsplib", n=n, xy=data["xy"], ew=ew, problem_file=str(pf),
                        optimum=int(optimum), optimum_source="test (exact MILP)" if TEST_MODE else "TSPLIB"))
    for inst in out:
        inst["cap"] = CAP_PER_NODE * inst["n"]
        inst["dir"] = OUTPUT_ROOT / "instances" / inst["name"]
        inst["dir"].mkdir(parents=True, exist_ok=True)
        inst["sha256"] = sha256_of(inst["problem_file"])
        inst["opt_tour"] = None
        pilot = Path(PILOT_DIR) / "instances" / inst["name"] / "opt_tour.npz" if PILOT_DIR else None
        if pilot is not None and pilot.exists():
            tour = np.load(pilot)["tour"].astype(np.int64)
            if T.is_valid_tour(tour, inst["n"]) and T.tour_length(inst["xy"], tour, inst["ew"]) == inst["optimum"]:
                inst["opt_tour"] = tour
            else:
                print(f"UYARI: {inst['name']}: pilotun optimal turu bu örneğe uymuyor; kapsama analizinde kullanılmaz.")
    assert len({x["name"] for x in out}) == len(out)
    return out


INSTANCES = build_instances()
BY_NAME = {inst["name"]: inst for inst in INSTANCES}
print(f"{len(INSTANCES)} örnek: " + ", ".join(f"{g} {sum(i['group'] == g for i in INSTANCES)}"
                                              for g in dict.fromkeys(i["group"] for i in INSTANCES))
      + f" | pilotun optimal turu olan: {sum(i['opt_tour'] is not None for i in INSTANCES)} | toplam üst sınır "
      f"{sum(i['cap'] for i in INSTANCES) / 60:.0f} dk/kol/tohum")

In [ ]:
# Deneyin işlevleri: A) altı ilk deneme, B) BetaTour ve listeler, C) koşular. Her sonuç Drive'a hemen yazılır.
PRINT_LOCK = threading.Lock()
SESSION_ID = uuid.uuid4().hex[:12]
TMP = str(WORK_ROOT / "lkh_tmp")


def say(*parts):
    with PRINT_LOCK:
        print(f"[{time.strftime('%H:%M:%S')}]", *parts, flush=True)


def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=float))
    os.replace(tmp, path)


def read_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None


def save_npz_atomic(path, **arrays):
    path = Path(path)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)


def log_error(label, exc):
    write_json_atomic(OUTPUT_ROOT / "errors" / f"{label}_{int(time.time())}.json",
                      dict(label=label, error=repr(exc), traceback=traceback.format_exc(), session=SESSION_ID,
                           utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())))


# ------------------------------------------------------------------------------------------ A: ilk denemeler
def first_done(inst):
    return (inst["dir"] / "first.npz").exists()


def stage_first(instances):
    """Her örnekte altı ilk deneme (POPMUSIC-5, tek deneme; pop kolunun aynı tohumlu ilk denemesi). Ana adımdaki
    koşularla aynı paralel yük altında ölçülür."""
    todo = [inst for inst in instances if not first_done(inst)]
    jobs = [(inst, j) for inst in todo for j in range(len(SEEDS))]
    results = {}
    with ThreadPoolExecutor(max_workers=PARALLEL) as pool:
        futures = {pool.submit(V.first_trial, LKH, TMP, SEEDS[j], inst["xy"], inst["problem_file"], inst["ew"],
                               inst["optimum"], K_LIST): (inst["name"], j) for inst, j in jobs}
        for fut in as_completed(futures):
            name, j = futures[fut]
            results[(name, j)] = fut.result()
    for inst in todo:
        firsts = [results[(inst["name"], j)] for j in range(len(SEEDS))]
        for f in firsts:
            assert f["cost"] >= inst["optimum"], f"{inst['name']}: ilk deneme optimumdan kısa (optimum yanlış)"
        save_npz_atomic(inst["dir"] / "first.npz", tours=np.stack([f["tour"] for f in firsts]),
                        costs=np.array([f["cost"] for f in firsts]), cand=np.stack([f["cand"] for f in firsts]),
                        alpha=np.stack([f["alpha"] for f in firsts]), dad=np.stack([f["dad"] for f in firsts]),
                        cpu=np.array([f["time_total"] for f in firsts]), wall=np.array([f["wall"] for f in firsts]))
    return len(todo)


def load_firsts(inst):
    F = np.load(inst["dir"] / "first.npz")
    return [dict(tour=F["tours"][j], cost=float(F["costs"][j]), cand=F["cand"][j], alpha=F["alpha"][j],
                 dad=F["dad"][j], time_total=float(F["cpu"][j])) for j in range(len(SEEDS))]


# --------------------------------------------------------------------------------- B: BetaTour ve listeler
def stage_lists(inst):
    """BetaTour altı ilk deneme turunu okur; yeniden başlatılan kolların aday dosyaları kurulur ve her kolun duraklama
    süresi hesaplanır. Kapsama (pilotun optimal turu varsa): her listenin kaçırdığı optimal kenarlar."""
    path = inst["dir"] / "lists.npz"
    if path.exists():
        return "önbellekte"
    firsts = load_firsts(inst)
    succ = np.stack([T.tour_succ_pred(f["tour"])[0] for f in firsts], axis=1).astype(np.int32)
    costs = np.array([f["cost"] for f in firsts], dtype=np.float64)
    pred = M.predict_tourfold(MODEL, inst["xy"], K_GRAPH, DEVICE, ens_succ=succ, ens_cost=costs)
    lists, build, rank, count_rank = V.arm_lists(firsts, pred, k=K_LIST, k_union=K_UNION, k_learned=K_LEARNED)
    cpu = np.array([f["time_total"] for f in firsts])
    pause = V.pauses(cpu, pred["t_total"], build)
    arrays = {"rank": rank.astype(np.int32), "count_rank": count_rank.astype(np.int32),
              "NB": np.asarray(pred["NB"], dtype=np.int32), "prob": np.asarray(pred["prob"], dtype=np.float32)}
    for arm in V.RESTARTED:
        arrays[f"cand_{arm}"] = np.stack([c for c, _, _ in lists[arm]]).astype(np.int32)
        arrays[f"alpha_{arm}"] = np.stack([a for _, a, _ in lists[arm]]).astype(np.int64)
    arrays["dad"] = np.stack([f["dad"] for f in firsts]).astype(np.int32)
    meta = dict(pause=pause, build_s=build, model_s=float(pred["t_total"]), first_cpu=cpu.tolist(),
                first_costs=costs.tolist(), first_found=[bool(c == inst["optimum"]) for c in costs])
    if inst["opt_tour"] is not None:
        missed = {arm: [V.missed_optimal_edges(c, inst["opt_tour"]) for c, _, _ in lists[arm]] for arm in V.RESTARTED}
        missed["pop"] = list(missed["pop_r"])           # pop'un listeleri: aynı tohumun POPMUSIC-5 listeleri
        al = T.task_candidates(LKH, TMP, problem_file=inst["problem_file"], xy=inst["xy"], ew=inst["ew"],
                               ctype="ALPHA", subgradient=True, k=K_LIST)
        missed["alpha"] = [V.missed_optimal_edges(al["cand"], inst["opt_tour"])] * len(SEEDS)
        meta["missed_optimal_edges"] = missed
    save_npz_atomic(path, **arrays)
    write_json_atomic(inst["dir"] / "lists.json", meta)
    return f"BetaTour {pred['t_total']:.2f} s, duraklama {pause['beta']:.1f} s"


LIST_CACHE, LIST_LOCK = {}, threading.Lock()


def lists_of(inst):
    with LIST_LOCK:
        if inst["name"] not in LIST_CACHE:
            Z = np.load(inst["dir"] / "lists.npz")
            LIST_CACHE[inst["name"]] = ({key: Z[key] for key in Z.files}, read_json(inst["dir"] / "lists.json"),
                                        np.load(inst["dir"] / "first.npz")["tours"])
        return LIST_CACHE[inst["name"]]


# ------------------------------------------------------------------------------------------------ C: koşular
def run_path(inst, arm, j):
    return OUTPUT_ROOT / "runs" / inst["name"] / f"{arm}_{SEEDS[j]}.json"


def run_job(inst, arm, j):
    """Bir koşu: optimumda ya da üst sınırda durur. Kayıt hemen Drive'a yazılır."""
    path = run_path(inst, arm, j)
    if path.exists():
        return read_json(path)
    cap = inst["cap"]
    rec = dict(instance=inst["name"], group=inst["group"], n=inst["n"], arm=arm, seed=SEEDS[j], cap=cap,
               optimum=inst["optimum"], session=SESSION_ID, cpu_model=HARDWARE["cpu_model"])
    if arm in V.UNINTERRUPTED:
        res = V.lkh_run(LKH, TMP, inst["problem_file"], inst["xy"], inst["ew"], SEEDS[j], inst["optimum"], cap,
                        params=dict(V.UNINTERRUPTED[arm], MAX_CANDIDATES=K_LIST))
        rec.update(res, pause=0.0, first_found=False)
    else:
        arrays, meta, tours = lists_of(inst)
        rec.update(pause=float(meta["pause"][arm]), first_cpu=float(meta["first_cpu"][j]),
                   first_found=bool(meta["first_found"][j]))
        remaining = cap - rec["pause"]
        if rec["first_found"]:
            rec.update(found=True, time_total=None, note="first trial reached the optimum")
        elif remaining <= 0.05:
            rec.update(found=False, time_total=None, note="pause longer than the cap")
        else:
            files = (arrays[f"cand_{arm}"][j], arrays[f"alpha_{arm}"][j], arrays["dad"][j])
            res = V.lkh_run(LKH, TMP, inst["problem_file"], inst["xy"], inst["ew"], SEEDS[j], inst["optimum"],
                            remaining, files=files, init_tour=tours[j])
            rec.update(res)
    if rec.get("shorter") or rec.get("new_optimum"):
        raise RuntimeError(f"{inst['name']} {arm} {SEEDS[j]}: optimumdan kısa tur bulundu; optimum değeri yanlış")
    if not rec["first_found"] and rec.get("time_total") is None and rec.get("note") is None:
        rec.update(found=False, note=f"LKH did not report Time.total (rc={rec.get('returncode')}, "
                                     f"timed out={rec.get('timed_out')})")
        log_error(f"{inst['name']}_{arm}_{SEEDS[j]}", RuntimeError(rec["note"] + " " + rec.get("stdout_tail", "")[-400:]))
    tto = V.time_to_optimum(rec, cap)
    rec["tto"] = tto if math.isfinite(tto) else None
    rec["utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    write_json_atomic(path, rec)
    return rec


def all_jobs():
    jobs = [(inst["name"], arm, j) for inst in INSTANCES for arm in ARMS for j in range(len(SEEDS))]
    order = np.random.default_rng(JOB_ORDER_SEED).permutation(len(jobs))
    return [jobs[i] for i in order]

In [ ]:
# Öz sınama (1-2 dk), ilk örnekten önce. Bu makinede şunları denetler:
#  1) pop kolunun tek denemelik koşusu, ilk deneme işleviyle aynı turu verir (aynı tohum);
#  2) OPTIMUM verilince LKH-3 o uzunlukta durur; ulaşılamaz OPTIMUM'da TOTAL_TIME_LIMIT'te kendisi durur ve turunu
#     yazar (işlemci saati);
#  3) BetaTour listeleriyle yeniden başlatma çalışır; 4) run v21'in kodlama öz sınaması (LKH-3'ün kendi dosyası ile
#  yeniden yazılmış dosya aynı aramayı verir; ağaç kuralı LKH-3'ün alfa = 0 girdilerini verir); 5) istatistik araçları.
SELF = {"utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "n": SELF_N}
xy_s = T.to_int_coords(T.gen_instance(SELF_N, "uniform", np.random.default_rng(424242)))
(WORK_ROOT / "problems").mkdir(exist_ok=True)
pf_s = str(WORK_ROOT / "problems" / "selfcheck.tsp")
T.write_tsp(pf_s, xy_s, name="selfcheck")
UNREACHABLE = 1                                      # hiçbir tur bu uzunlukta değildir

first_s = V.first_trial(LKH, TMP, SEEDS[0], xy_s, pf_s, "EUC_2D", UNREACHABLE, K_LIST)
one = V.lkh_run(LKH, TMP, pf_s, xy_s, "EUC_2D", SEEDS[0], UNREACHABLE, 60.0,
                params=dict(V.UNINTERRUPTED["pop"], MAX_CANDIDATES=K_LIST), max_trials=1, return_tour=True)
SELF["first_trial_equals_pop_first_trial"] = bool(one["tour"] is not None
                                                  and np.array_equal(one["tour"], first_s["tour"].astype(np.int64)))

probe = [V.lkh_run(LKH, TMP, pf_s, xy_s, "EUC_2D", s, UNREACHABLE, 2.0, params=dict(V.UNINTERRUPTED["pop"]))
         for s in (11, 12)]
best = min(probe, key=lambda r: r["cost"])
best_seed = (11, 12)[probe.index(best)]
stop = V.lkh_run(LKH, TMP, pf_s, xy_s, "EUC_2D", best_seed, int(best["cost"]), 2.0,
                 params=dict(V.UNINTERRUPTED["pop"]))
SELF["stop_at_optimum"] = dict(target=best["cost"], found=stop["found"], time_total=stop["time_total"],
                               probe_time_total=best["time_total"])
SELF["time_limit"] = dict(limit=2.0, time_totals=[r["time_total"] for r in probe],
                          returncodes=[r["returncode"] for r in probe])

firsts_s = [V.first_trial(LKH, TMP, s, xy_s, pf_s, "EUC_2D", UNREACHABLE, K_LIST) for s in SEEDS]
succ_s = np.stack([T.tour_succ_pred(f["tour"])[0] for f in firsts_s], axis=1).astype(np.int32)
pred_s = M.predict_tourfold(MODEL, xy_s, K_GRAPH, DEVICE, ens_succ=succ_s,
                            ens_cost=np.array([f["cost"] for f in firsts_s]))       # GPU ısınması
pred_s = M.predict_tourfold(MODEL, xy_s, K_GRAPH, DEVICE, ens_succ=succ_s,
                            ens_cost=np.array([f["cost"] for f in firsts_s]))
lists_s, build_s, _, _ = V.arm_lists(firsts_s, pred_s, k=K_LIST, k_union=K_UNION, k_learned=K_LEARNED)
restart = {}
for arm in V.RESTARTED:
    r = V.lkh_run(LKH, TMP, pf_s, xy_s, "EUC_2D", SEEDS[0], UNREACHABLE, 1.0, files=lists_s[arm][0],
                  init_tour=firsts_s[0]["tour"])
    restart[arm] = dict(cost=r["cost"], returncode=r["returncode"], time_total=r["time_total"],
                        not_longer_than_first=bool(r["cost"] is not None and r["cost"] <= firsts_s[0]["cost"]))
SELF["restart"] = restart
SELF["model_s"] = float(pred_s["t_total"])
SELF["list_build_s"] = build_s
SELF["encoding"] = A.encoding_selfcheck(LKH, TMP, xy_s, SEEDS[0], trials=25)
SELF["statistics"] = dict(holm=V.holm([0.01, 0.04]).tolist(), compare=V.compare([1, 2, 3, 5], [2, 2, 4, 4]),
                          par=V.par([1.0, float("inf")], 10.0))
write_json_atomic(OUTPUT_ROOT / "selfcheck" / "selfcheck.json", SELF)

checks = {
    "tek denemelik pop koşusu = ilk deneme": SELF["first_trial_equals_pop_first_trial"],
    "OPTIMUM'da durur": stop["found"] and stop["time_total"] is not None and stop["time_total"] <= best["time_total"] + 0.05,
    "işlemci saati sınırında kendisi durur": all(r["returncode"] == 0 and r["time_total"] is not None
                                                 and 1.9 <= r["time_total"] <= 2.6 for r in probe),
    "yeniden başlatmalar çalışır": all(v["returncode"] == 0 and v["not_longer_than_first"] for v in restart.values()),
    "kendi dosyası = LKH dosyası (run v21)": SELF["encoding"]["native_file_equals_lkh_file"],
    "ağaç kuralı (run v21)": SELF["encoding"]["tree_rule_matches_lkh"],
    "Holm ve işaret testi": np.allclose(SELF["statistics"]["holm"], [0.02, 0.04])
    and SELF["statistics"]["compare"]["wins"] == 2 and SELF["statistics"]["compare"]["losses"] == 1
    and abs(SELF["statistics"]["par"] - 50.5) < 1e-9,
}
SELF["checks"] = {k: bool(v) for k, v in checks.items()}
write_json_atomic(OUTPUT_ROOT / "selfcheck" / "selfcheck.json", SELF)
for k, v in checks.items():
    print(("  tamam  " if v else "  HATA   ") + k)
print(f"BetaTour çıkarımı {SELF['model_s']:.2f} s ({SELF_N} düğüm); OPTIMUM denemesi: hedef {best['cost']:.0f}, "
      f"durduğu süre {stop['time_total']} s (deneme koşusunda {best['time_total']} s).")
assert all(checks.values()), "Öz sınama geçmedi; selfcheck/selfcheck.json dosyasını gönderin."

In [ ]:
# Ana adım. A) ilk denemeler, B) BetaTour ve listeler, C) bütün koşular tek rastgele sırada (PARALLEL süreç aynı
# anda). Biten her koşu Drive'a yazılır; defter yeniden çalıştırılınca kalanlardan devam eder.
T0 = time.time()
left_first = sum(not first_done(inst) for inst in INSTANCES)
say(f"A) İlk denemeler: {left_first} örnek x {len(SEEDS)} (tahmini {max(1, round(left_first * 6 * 2 / PARALLEL / 60))}"
    "-3 dk)")
stage_first(INSTANCES)
say(f"B) BetaTour ve listeler ({DEVICE}; tahmini 2-4 dk)")
for inst in INSTANCES:
    say(f"B: {inst['name']}: {stage_lists(inst)}")

JOBS = all_jobs()
todo = [job for job in JOBS if not run_path(BY_NAME[job[0]], job[1], job[2]).exists()]
cap_hours = sum(BY_NAME[name]["cap"] for name, _, _ in todo) / 3600
say(f"C) {len(todo)}/{len(JOBS)} koşu kaldı, aynı anda {PARALLEL} süreç. Üst sınırların toplamı {cap_hours:.1f} "
    f"çekirdek-saat; tahmini süre {0.35 * cap_hours / PARALLEL:.1f}-{0.7 * cap_hours / PARALLEL:.1f} saat "
    "(ilk 50 koşudan sonra yeniden tahmin edilir).")
t_c, finished, failed = time.time(), 0, 0
with ThreadPoolExecutor(max_workers=PARALLEL) as pool:
    futures = {pool.submit(run_job, BY_NAME[name], arm, j): (name, arm, j) for name, arm, j in todo}
    for fut in as_completed(futures):
        name, arm, j = futures[fut]
        try:
            fut.result()
            finished += 1
        except Exception as exc:
            failed += 1
            log_error(f"{name}_{arm}_{SEEDS[j]}", exc)
            say(f"HATA: {name} {arm} {SEEDS[j]}: {exc!r}")
        done = finished + failed
        if done % 50 == 0 or done == len(todo):
            elapsed = time.time() - t_c
            eta = (len(todo) - done) * elapsed / max(done, 1)
            say(f"C: {done}/{len(todo)} koşu ({failed} hata); geçen {elapsed / 60:.0f} dk, kalan tahmini "
                f"{eta / 3600:.1f} saat")
missing = [job for job in JOBS if not run_path(BY_NAME[job[0]], job[1], job[2]).exists()]
say(f"Bitti: {len(JOBS) - len(missing)}/{len(JOBS)} koşu kayıtlı; bu oturum {(time.time() - T0) / 3600:.2f} saat."
    + (f" Eksik {len(missing)} koşu var: defteri yeniden çalıştırın (errors/ klasörüne bakın)." if missing else ""))

In [ ]:
# Analiz ve karar (saniyeler). Birincil: H1 (beta < pop) ve H2 (beta < alpha), PAR10 üzerinde örnek bazında kesin
# işaret testi, Holm. Geri kalanı betimseldir. Tablolar ekrana yazılır ve summary/ klasörüne kaydedilir.
INF = float("inf")
records = [read_json(run_path(inst, arm, j)) for inst in INSTANCES for arm in ARMS for j in range(len(SEEDS))]
MISSING = sum(r is None for r in records)
RUNS = pd.DataFrame([r for r in records if r is not None])
RUNS["success"] = RUNS["tto"].notna()

rows = []
for inst in INSTANCES:
    for arm in ARMS:
        sub = RUNS[(RUNS["instance"] == inst["name"]) & (RUNS["arm"] == arm)]
        times = [INF if t is None or (isinstance(t, float) and math.isnan(t)) else float(t) for t in sub["tto"]]
        rows.append(dict(instance=inst["name"], group=inst["group"], n=inst["n"], arm=arm, cap=inst["cap"],
                         runs=len(sub), successes=int(sub["success"].sum()),
                         par10=V.par(times, inst["cap"], PAR_FACTOR) if times else np.nan,
                         par2=V.par(times, inst["cap"], 2.0) if times else np.nan,
                         median=float(np.median(times)) if times else np.nan,
                         pause=float(sub["pause"].mean()) if len(sub) else np.nan))
PER = pd.DataFrame(rows)
COMPLETE = [inst["name"] for inst in INSTANCES
            if (PER[PER["instance"] == inst["name"]]["runs"] == len(SEEDS)).all()]
WIDE = PER.pivot(index="instance", columns="arm", values="par10").loc[COMPLETE, ARMS]
GROUP_OF = {inst["name"]: inst["group"] for inst in INSTANCES}


CAP_OF = {inst["name"]: inst["cap"] for inst in INSTANCES}


def paired(first, second, names=None, seed=BOOTSTRAP_SEED):
    names = list(WIDE.index) if names is None else names
    tol = np.array([TIE_FRACTION * CAP_OF[name] for name in names])
    out = V.compare(WIDE.loc[names, first], WIDE.loc[names, second], tol=tol)
    out.update(V.ratio_interval(WIDE.loc[names, first], WIDE.loc[names, second], seed))
    out["instances"] = len(names)
    return out


h1, h2 = paired("beta", "pop"), paired("beta", "alpha")
adjusted = V.holm([h1["p"], h2["p"]])
h1["p_holm"], h2["p_holm"] = float(adjusted[0]), float(adjusted[1])
H1 = bool(h1["wins"] > h1["losses"] and h1["p_holm"] < 0.05)
H2 = bool(h2["wins"] > h2["losses"] and h2["p_holm"] < 0.05)
SECONDARY = paired("beta", "count")
CREDITED = bool(SECONDARY["wins"] > SECONDARY["losses"] and SECONDARY["p"] < 0.05)
if H1 and H2:
    VERDICT = ("H1 ve H2 desteklendi: BetaTour listeleriyle LKH-3 kanıtlı optimuma kendi listelerinden daha çabuk "
               "ulaşıyor. " + ("Öğrenilmiş sıralamaya pay verilir (beta, sayma listesini de yeniyor)."
                               if CREDITED else "Kazanç, ilk deneme turlarından kurulan listelere verilir (beta, sayma "
                                                "listesinden belirgin biçimde iyi değil)."))
elif H1 or H2:
    VERDICT = f"Kısmi sonuç: yalnızca {'H1 (pop)' if H1 else 'H2 (alpha)'} desteklendi."
else:
    VERDICT = "H1 ve H2 desteklenmedi: optimuma ulaşma süresinde üstünlük yok (olumsuz sonuç olarak raporlanır)."

PAIRS = [("beta", "pop"), ("beta", "alpha"), ("beta", "count"), ("beta", "pop_r"), ("pop_r", "pop"),
         ("mix", "beta"), ("union", "beta"), ("count", "pop"), ("count", "alpha"), ("mix", "pop"), ("union", "pop"),
         ("alpha", "pop")]
groups = list(dict.fromkeys(GROUP_OF[name] for name in WIDE.index))
tsplib_names = [name for name in WIDE.index if GROUP_OF[name] == "tsplib"]
synthetic_names = [name for name in WIDE.index if GROUP_OF[name] != "tsplib"]
pair_rows = []
for a, b in PAIRS:
    for label, names in [("hepsi", list(WIDE.index)), ("sentetik", synthetic_names), ("TSPLIB", tsplib_names)]:
        if len(names) >= 2:
            r = paired(a, b, names)
            pair_rows.append({"karşılaştırma": f"{a} - {b}", "örnekler": label, "n": r["instances"],
                              "kazanç/eşit/kayıp": f"{r['wins']}/{r['ties']}/{r['losses']}", "p (düzeltmesiz)": r["p"],
                              "PAR10 oranı": r["ratio"], "%95 aralık": f"{r['low']:.3f}-{r['high']:.3f}"})
PAIR_TABLE = pd.DataFrame(pair_rows)

success = PER[PER["instance"].isin(WIDE.index)].groupby(["group", "arm"])["successes"].sum().unstack()[ARMS]
runs_per_group = PER[PER["instance"].isin(WIDE.index)].groupby(["group", "arm"])["runs"].sum().unstack()[ARMS]
SUCCESS_TABLE = (100.0 * success / runs_per_group).loc[groups]
GEO_TABLE = PER[PER["instance"].isin(WIDE.index)].groupby(["group", "arm"])["par10"].apply(
    lambda s: float(np.exp(np.log(s).mean()))).unstack()[ARMS].loc[groups]
PAUSE_TABLE = PER[PER["arm"].isin(V.RESTARTED)].groupby(["group", "arm"])["pause"].mean().unstack()[list(V.RESTARTED)]

coverage_rows = []
for inst in INSTANCES:
    meta = read_json(inst["dir"] / "lists.json") or {}
    missed = meta.get("missed_optimal_edges")
    if not missed:
        continue
    for arm in ARMS:
        for j, seed in enumerate(SEEDS):
            row = RUNS[(RUNS["instance"] == inst["name"]) & (RUNS["arm"] == arm) & (RUNS["seed"] == seed)]
            if len(row):
                coverage_rows.append(dict(instance=inst["name"], arm=arm, seed=seed, missed=int(missed[arm][j]),
                                          success=bool(row["success"].iloc[0])))
COVERAGE = pd.DataFrame(coverage_rows)
if len(COVERAGE):
    COVERAGE_TABLE = COVERAGE.assign(whole_tour=COVERAGE["missed"] == 0).groupby(["arm", "whole_tour"]).agg(
        koşu=("success", "size"), başarı_yüzdesi=("success", lambda s: 100.0 * s.mean())).unstack()
    MISSED_TABLE = COVERAGE.groupby("arm")["missed"].agg(["mean", "max"]).loc[ARMS]
else:
    COVERAGE_TABLE = MISSED_TABLE = pd.DataFrame()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
print(f"Kayıtlı koşu: {len(RUNS)}/{len(records)} (eksik {MISSING}); tam örnek: {len(WIDE)}/{len(INSTANCES)}")
print("\nTablo 1. Optimuma ulaşan koşuların yüzdesi (gruplara göre)")
print(SUCCESS_TABLE.round(1).to_string())
print("\nTablo 2. PAR10 (saniye; örnekler üzerinde geometrik ortalama)")
print(GEO_TABLE.round(2).to_string())
print("\nTablo 3. Duraklama (saniye; yeniden başlatılan kollar)")
print(PAUSE_TABLE.round(2).to_string())
print("\nTablo 4. Eşli karşılaştırmalar (PAR10, örnek bazında; oran < 1: ilk kol daha çabuk)")
print(PAIR_TABLE.round(4).to_string(index=False))
if len(COVERAGE):
    print("\nTablo 5. Pilotun optimal turları olan örneklerde: listelerin kaçırdığı optimal kenar (koşu başına) ve "
          "optimal turun tamamı listelerdeyken / değilken başarı")
    print(MISSED_TABLE.round(2).to_string())
    print(COVERAGE_TABLE.round(1).to_string())
print("\nKarar kuralı (protokol)")
print(f"  H1 beta < pop  : {h1['wins']}/{h1['ties']}/{h1['losses']}, p = {h1['p']:.4g}, Holm p = {h1['p_holm']:.4g}, "
      f"PAR10 oranı {h1['ratio']:.3f} ({h1['low']:.3f}-{h1['high']:.3f}) -> {'DESTEKLENDİ' if H1 else 'desteklenmedi'}")
print(f"  H2 beta < alpha: {h2['wins']}/{h2['ties']}/{h2['losses']}, p = {h2['p']:.4g}, Holm p = {h2['p_holm']:.4g}, "
      f"PAR10 oranı {h2['ratio']:.3f} ({h2['low']:.3f}-{h2['high']:.3f}) -> {'DESTEKLENDİ' if H2 else 'desteklenmedi'}")
print(f"  İkincil beta < count: {SECONDARY['wins']}/{SECONDARY['ties']}/{SECONDARY['losses']}, "
      f"p = {SECONDARY['p']:.4g} (düzeltmesiz)")
print("  Sonuç:", VERDICT)

out_dir = OUTPUT_ROOT / "summary"
RUNS.drop(columns=[c for c in ["stdout_tail"] if c in RUNS.columns]).to_csv(out_dir / "runs.csv", index=False)
PER.to_csv(out_dir / "per_instance_arm.csv", index=False)
WIDE.to_csv(out_dir / "par10_wide.csv")
PAIR_TABLE.to_csv(out_dir / "pairs.csv", index=False)
if len(COVERAGE):
    COVERAGE.to_csv(out_dir / "coverage_runs.csv", index=False)
write_json_atomic(out_dir / "summary.json", {
    "decision": {"H1": H1, "H2": H2, "h1": h1, "h2": h2, "secondary_beta_vs_count": SECONDARY,
                 "learned_ranking_credited": CREDITED, "verdict": VERDICT},
    "complete_instances": len(WIDE), "instances": len(INSTANCES), "missing_runs": MISSING,
    "success_pct": SUCCESS_TABLE.reset_index().to_dict(orient="records"),
    "par10_geomean": GEO_TABLE.reset_index().to_dict(orient="records"),
    "pause_s": PAUSE_TABLE.reset_index().to_dict(orient="records"),
    "pairs": PAIR_TABLE.to_dict(orient="records"),
    "missed_optimal_edges": MISSED_TABLE.reset_index().to_dict(orient="records") if len(COVERAGE) else None,
    "selfcheck": SELF, "manifest": MANIFEST})
print(f"\nKaydedildi: {out_dir}")

In [ ]:
# Arşiv (1 dk): koşu kayıtları, ilk denemeler (turlar ve listeler), liste özetleri, tablolar, öz sınama, kod ve
# manifest tek ZIP'te. Bunu gönderin. (lists.npz dosyaları büyük olduğu için ZIP'e girmez; Drive'da kalır.)
import zipfile


def make_zip(name, include):
    """ZIP önce yerel diskte kurulur, sonra sonuç klasörüne (Drive) kopyalanır."""
    local = WORK_ROOT / name
    count = 0
    with zipfile.ZipFile(local, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            relative = path.relative_to(OUTPUT_ROOT)
            if not path.is_file() or path.suffix in {".zip", ".tmp"} or path.name.endswith(".tmp.npz"):
                continue
            if include(relative):
                z.write(path, str(relative))
                count += 1
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None
    digest = hashlib.sha256(local.read_bytes()).hexdigest()
    shutil.copy2(local, OUTPUT_ROOT / name)
    print(f"{name}: {count} dosya, {local.stat().st_size / 1e6:.1f} MB, SHA-256 {digest}")
    return local


result_zip = make_zip(EXPERIMENT_NAME + "_results.zip",
                      lambda rel: rel.parts[0] in {"runs", "summary", "code", "errors", "selfcheck"}
                      or (rel.parts[0] == "instances" and rel.name in {"first.npz", "lists.json"})
                      or rel.name == "manifest.json")
if SAVE_TO_DRIVE:
    from google.colab import files
    files.download(str(result_zip))      # tarayıcıya iner
print("ZIP şu klasörde:", OUTPUT_ROOT)